# RAG Prompt Injection Assessment Workflow: Solution

This notebook tests prompt injection payloads embedded inside indexed customer support documents.

In [ ]:
from pathlib import Path
import sys

CURRENT = Path.cwd().resolve()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
sys.path.append(str(ROOT / "src"))

from rag_prompt_injection_utils import (
    GuardedRAGAssistant,
    VulnerableRAGAssistant,
    attack_success_rate,
    build_vector_store,
    create_poisoned_corpus,
    evaluate_attacks,
    load_json,
    load_text,
    summarize_results,
    write_results_csv,
)

DATA_DIR = ROOT / "data"
PROMPT_DIR = ROOT / "prompts"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

documents = load_json(DATA_DIR / "support_documents.json")
queries = load_json(DATA_DIR / "evaluation_queries.json")
payloads = load_json(DATA_DIR / "prompt_injection_payload_templates.json")
system_prompt = load_text(PROMPT_DIR / "rag_system_prompt.md")

print(f"Documents: {len(documents)}")
print(f"Evaluation queries: {len(queries)}")
print(f"Payload templates: {len(payloads)}")

## 1. Build a Clean FAISS-Style Retrieval Index

In [ ]:
clean_store = build_vector_store(documents)
for query in queries[:2]:
    retrieved = clean_store.search(query["query"], k=2)
    print(query["id"], [doc.id for doc in retrieved])

## 2. Insert Prompt Injection Payloads Into Indexed Documents

In [ ]:
poisoned_documents = create_poisoned_corpus(documents, payloads)
poisoned_store = build_vector_store(poisoned_documents)

for payload in payloads:
    print(f"Inserted {payload['id']} into {payload['target_doc_id']}")

## 3. Run Attacks Against the Vulnerable RAG Assistant

In [ ]:
vulnerable_assistant = VulnerableRAGAssistant(system_prompt, poisoned_store, top_k=2)
attack_results = evaluate_attacks(vulnerable_assistant, queries, payloads)

write_results_csv(attack_results, RESULTS_DIR / "solution_attack_results.csv")
print(f"Attack success rate: {attack_success_rate(attack_results):.2f}")
summarize_results(attack_results)

## 4. Compare Against a Guarded RAG Assistant

In [ ]:
guarded_assistant = GuardedRAGAssistant(system_prompt, poisoned_store, top_k=2)
defended_results = evaluate_attacks(guarded_assistant, queries, payloads)

write_results_csv(defended_results, RESULTS_DIR / "solution_defended_results.csv")
print(f"Defended attack success rate: {attack_success_rate(defended_results):.2f}")
summarize_results(defended_results)

## 5. Findings

The vulnerable assistant follows injected instructions when poisoned documents are retrieved. The guarded assistant detects instruction-like text in retrieved content, blocks the injected document instructions, and returns policy-aligned answers. In production, this should be combined with stronger retrieval filtering, content provenance, least-privilege indexing, output validation, and monitoring.